# L8d: Classification of Consumer Credit Score
In this lab, we will construct, train, and evaluate a feedforward neural network to classify consumer credit scores.

* _What is a credit score?_ A credit score is a numerical expression based on a person's credit files that represents creditworthiness. It is primarily based on credit report information sourced from credit bureaus. Lenders use the score to evaluate the risk of lending money to consumers; the score ranges from 300 to 850, where higher scores indicate lower risk.
* _Hypothetical scenario:_ You are a data scientist at a global manufacturing company. Your credit division has collected basic bank details and credit-related information on customers who finance product purchases. You have been tasked with building a system to classify customers into credit score brackets to reduce manual effort and speed up loan processing.

> __Learning Objectives__
>
> By the end of this lab, you will be able to:
> * __Prepare and encode tabular data for classification:__ Encode categorical variables as integers and apply z-score standardization to continuous features. Convert the processed records into the `Vector{Tuple{Vector{Float32}, OneHotVector{UInt32}}}` format required by `Flux.jl`.
> * __Construct and train a feedforward neural network classifier:__ Build a multi-layer feedforward network using `Flux.jl` with `Dense` layers and `tanh` activation. Train the network by minimizing a logit cross-entropy loss using gradient descent with momentum.
> * __Evaluate model generalization:__ Compute classification accuracy on both training and test datasets. Use the difference between training and test accuracy to assess whether the model overfits.

Let's get started!
___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, and setting up any required constants.

> __Environment Setup with Include.jl__
>
> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
include("Include.jl"); # This will load necessary packages and functions

### Data
Suppose we have a dataset $\mathcal{D} = \left\{(\mathbf{x}_{1},y_{1}),(\mathbf{x}_{2},y_{2}),\dots,(\mathbf{x}_{n},y_{n})\right\}$ containing $n$ records, where each record is a tuple $(\mathbf{x}, y)$, where $\mathbf{x} \in \mathbb{R}^m$ is a vector of features and $y \in \mathcal{Y}$ is a label. The label $y$ is a categorical variable: $\mathcal{Y} \equiv \left\{\texttt{Poor},\texttt{Standard},\texttt{Good}\right\}$.

The raw dataset is a CSV file with approximately 99k records. Each record contains `20` feature variables related to consumer credit, such as income, age, and other attributes that may influence credit risk, plus `1` label variable (`Credit_Score`). The label variable indicates whether the consumer has a `Poor`, `Standard`, or `Good` credit score. The dataset is [available on Kaggle](https://www.kaggle.com/datasets/sudhanshu2198/processed-data-credit-score?resource=download).

The code block below returns `raw_data::DataFrame`.

In [2]:
raw_data = CSV.read(joinpath(_PATH_TO_DATA, "credit_score_dataset.csv"), DataFrame)

Row,Delay_from_due_date,Num_of_Delayed_Payment,Num_Credit_Inquiries,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Amount_invested_monthly,Monthly_Balance,Credit_Score,Credit_Mix,Payment_Behaviour,Age,Annual_Income,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Monthly_Inhand_Salary,Changed_Credit_Limit,Outstanding_Debt,Total_EMI_per_month
,Float64,Float64,Float64,Float64,Float64,String3,Float64,Float64,String15,String15,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,3.0,7.0,4.0,26.8226,265.0,No,80.4153,312.494,Good,Good,High_spent_Medium_value_payments,23.0,19114.1,3.0,4.0,3.0,4.0,1824.84,11.27,809.98,49.5749
2,3.0,7.0,4.0,31.945,265.0,No,118.28,284.629,Good,Good,High_spent_Medium_value_payments,23.0,19114.1,3.0,4.0,3.0,4.0,1824.84,11.27,809.98,49.5749
3,3.0,7.0,4.0,28.6094,267.0,No,81.6995,331.21,Good,Good,High_spent_Medium_value_payments,23.0,19114.1,3.0,4.0,3.0,4.0,1824.84,11.27,809.98,49.5749
4,5.0,4.0,4.0,31.3779,268.0,No,199.458,223.451,Good,Good,High_spent_Medium_value_payments,23.0,19114.1,3.0,4.0,3.0,4.0,1824.84,11.27,809.98,49.5749
5,6.0,4.0,4.0,24.7973,269.0,No,41.4202,341.489,Good,Good,High_spent_Medium_value_payments,23.0,19114.1,3.0,4.0,3.0,4.0,1824.84,11.27,809.98,49.5749
6,8.0,4.0,4.0,27.2623,270.0,No,62.4302,340.479,Good,Good,High_spent_Medium_value_payments,23.0,19114.1,3.0,4.0,3.0,4.0,1824.84,11.27,809.98,49.5749
7,3.0,8.0,4.0,22.5376,271.0,No,178.344,244.565,Good,Good,High_spent_Medium_value_payments,23.0,19114.1,3.0,4.0,3.0,4.0,1824.84,11.27,809.98,49.5749
8,3.0,6.0,4.0,23.9338,271.0,No,24.7852,358.124,Standard,Good,High_spent_Medium_value_payments,23.0,19114.1,3.0,4.0,3.0,4.0,1824.84,11.27,809.98,49.5749
9,3.0,4.0,2.0,24.464,319.0,No,104.292,470.691,Standard,Good,High_spent_Large_value_payments,28.0,34847.8,2.0,4.0,6.0,1.0,3037.99,5.42,605.03,18.8162


Next, let's convert categorical variables to numerical variables and z-score standardize the numerical features.

> __What is going on in this code block?__
>
> We map `Payment_of_Min_Amount` to `No` $\rightarrow$ `-1` and `Yes` $\rightarrow$ `1`. We map `Credit_Score` to `Poor` $\rightarrow$ `1`, `Standard` $\rightarrow$ `2`, and `Good` $\rightarrow$ `3`. We map `Credit_Mix` with the same three-level integer encoding. We map `Payment_Behaviour` to six integer levels corresponding to combinations of spending amount and payment value. All remaining numerical variables are z-score standardized.

The code block below returns `dataset::DataFrame`.

In [3]:
dataset = let
    
    dataset = copy(raw_data); # make a copy of the raw data, so we can keep the original intact
    transform!(dataset, :Payment_of_Min_Amount => ByRow(x -> (x=="No" ? -1 : 1)) => :Payment_of_Min_Amount); # maps Payment_of_Min_Amount to -1,1
    transform!(dataset, :Credit_Score => ByRow(s -> convertcreditscore(s))  => :Credit_Score); # maps Credit_Score to 1,2,3
    transform!(dataset, :Credit_Mix => ByRow(s -> convertcreditmix(s))  => :Credit_Mix); # maps Credit_Mix to 1,2,3
    transform!(dataset, :Payment_Behaviour => ByRow(s -> convertcreditbehavior(s))  => :Payment_Behaviour); # maps Payment_Behaviour to 1,2,3,4,5,6
    categorical_fields = ["Credit_Mix", "Payment_Behaviour", "Credit_Score", "Payment_of_Min_Amount"];

    # names to convert -
    all_names = names(dataset)
    names_to_convert = Set{String}()
    for name in all_names
        if (in(name, categorical_fields) == false)
            push!(names_to_convert, name)
        end
    end

    # z-score standardization
    for name in names_to_convert
        name_symbol = Symbol(name);
        μ = mean(dataset[!, name_symbol])
        σ = std(dataset[!, name_symbol])
        transform!(dataset, name_symbol => ByRow(x -> (x - μ) / σ) => name_symbol); # standardize Delay_from_due_date
    end

    # convert categorical fields to categorical    
    dataset;
end

Row,Delay_from_due_date,Num_of_Delayed_Payment,Num_Credit_Inquiries,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Amount_invested_monthly,Monthly_Balance,Credit_Score,Credit_Mix,Payment_Behaviour,Age,Annual_Income,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Monthly_Inhand_Salary,Changed_Credit_Limit,Outstanding_Debt,Total_EMI_per_month
,Float64,Float64,Float64,Float64,Float64,Int64,Float64,Float64,Int64,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,-1.22042,-1.01059,-0.459468,-1.06743,0.440109,-1,-0.581417,-0.424237,3,1,5,-0.954179,-0.819564,-0.914032,-0.741333,-1.31966,0.190514,-0.744377,0.134091,-0.53368,-0.445004
2,-1.22042,-1.01059,-0.459468,-0.0663653,0.440109,-1,-0.387021,-0.554212,3,1,5,-0.954179,-0.819564,-0.914032,-0.741333,-1.31966,0.190514,-0.744377,0.134091,-0.53368,-0.445004
3,-1.22042,-1.01059,-0.459468,-0.718247,0.46017,-1,-0.574824,-0.336938,3,1,5,-0.954179,-0.819564,-0.914032,-0.741333,-1.31966,0.190514,-0.744377,0.134091,-0.53368,-0.445004
4,-1.08554,-1.48906,-0.459468,-0.177194,0.470201,-1,0.0297401,-0.839574,3,1,5,-0.954179,-0.819564,-0.914032,-0.741333,-1.31966,0.190514,-0.744377,0.134091,-0.53368,-0.445004
5,-1.0181,-1.48906,-0.459468,-1.46323,0.480231,-1,-0.781615,-0.288991,3,1,5,-0.954179,-0.819564,-0.914032,-0.741333,-1.31966,0.190514,-0.744377,0.134091,-0.53368,-0.445004
6,-0.88321,-1.48906,-0.459468,-0.981512,0.490262,-1,-0.673751,-0.293702,3,1,5,-0.954179,-0.819564,-0.914032,-0.741333,-1.31966,0.190514,-0.744377,0.134091,-0.53368,-0.445004
7,-1.22042,-0.851097,-0.459468,-1.90486,0.500292,-1,-0.0786576,-0.741088,3,1,5,-0.954179,-0.819564,-0.914032,-0.741333,-1.31966,0.190514,-0.744377,0.134091,-0.53368,-0.445004
8,-1.22042,-1.17008,-0.459468,-1.632,0.500292,-1,-0.867017,-0.211398,2,1,5,-0.954179,-0.819564,-0.914032,-0.741333,-1.31966,0.190514,-0.744377,0.134091,-0.53368,-0.445004
9,-1.22042,-1.48906,-0.977305,-1.52837,0.981756,-1,-0.458836,0.313664,2,1,6,-0.489597,-0.4087,-1.29988,-0.741333,-0.976448,-1.0359,-0.363666,-0.76441,-0.711087,-0.689468


Next, let's package the dataset into a `Vector{Tuple{Vector{Float32}, OneHotVector{UInt32}}}` data structure.

> __What is going on in this code block?__
>
> This format is a direct code representation of the dataset $\mathcal{D}$. The `Flux.jl` training loop accepts this format directly, making training convenient. All feature values are converted to `Float32`, and each label is encoded as a `OneHotVector`.

The code block below returns `(converted_dataset, features)`.

In [4]:
converted_dataset, features = let
    converted_dataset = Vector{Tuple{Vector{Float32}, OneHotVector{UInt32}}}();
    number_digit_array = [1,2,3]; # this is the digits for the labels

    # which cols do we want to use as features - let's use all *but* the label -
    all_cols = names(dataset);
    features = Array{String,1}();
    for col in all_cols
        if col != "Credit_Score"  # names() returns Strings, so compare to a String (not a Symbol)
            push!(features, col);
        end
    end
    features = features |> sort;
    number_of_features = length(features);
    
    # build record tuples -
    for record ∈ eachrow(dataset)

        # convert the label to a one-hot vector
        label = record[:Credit_Score]; # this is the label we want to predict 
        Y = onehot(label, number_digit_array); # convert the label to a one-hot vector
        X = Vector{Float32}();

        for i ∈ eachindex(features)
            feature = features[i]; # get the feature name
            value = record[feature] |> Float32; # get the value of the feature
            push!(X, value); # add the value to the feature vector
        end

        data_tuple = (X, Y); # create a tuple of the feature vector and the label
        push!(converted_dataset, data_tuple); # add the tuple to the dataset
    end

    converted_dataset, features;
end;

In [5]:
converted_dataset

99960-element Vector{Tuple{Vector{Float32}, OneHotVector{UInt32}}}:
 ([-0.9541789, -0.5814166, -0.81956404, 0.13409114, 0.44010937, 1.0, -1.0674309, -1.2204231, -1.3196558, -0.42423734, -0.744377, -0.91403186, -0.74133325, -0.45946833, -1.0105871, 0.19051377, -0.53367984, 5.0, -1.0, -0.4450041], [0, 0, 1])
 ([-0.9541789, -0.38702095, -0.81956404, 0.13409114, 0.44010937, 1.0, -0.06636529, -1.2204231, -1.3196558, -0.55421215, -0.744377, -0.91403186, -0.74133325, -0.45946833, -1.0105871, 0.19051377, -0.53367984, 5.0, -1.0, -0.4450041], [0, 0, 1])
 ([-0.9541789, -0.5748235, -0.81956404, 0.13409114, 0.46017033, 1.0, -0.7182474, -1.2204231, -1.3196558, -0.33693838, -0.744377, -0.91403186, -0.74133325, -0.45946833, -1.0105871, 0.19051377, -0.53367984, 5.0, -1.0, -0.4450041], [0, 0, 1])
 ([-0.9541789, 0.02974009, -0.81956404, 0.13409114, 0.47020084, 1.0, -0.17719401, -1.0855378, -1.3196558, -0.8395738, -0.744377, -0.91403186, -0.74133325, -0.45946833, -1.4890587, 0.19051377, -0.53367984, 5.0, 

### Constants
Set constants that we will use throughout the problem. See the comment next to each constant for a description, permissible values, and defaults.

In [6]:
θ = 0.60; # what fraction of the data to use for training. Default is 60%
number_of_training_samples = Int64(θ * length(converted_dataset)); # θ of the data will be used for training
number_of_test_samples = length(converted_dataset) - number_of_training_samples; # the rest will be used for testing
number_of_features = length(features); # the number of features, i.e, the length of x
number_of_classes = 3; # the number of classes for the problem {Poor | Standard | Good}
number_of_epochs = 50; # how many epochs do we want to train for?
number_of_hidden_nodes = 2^12; # TODO: Change the width of the hidden layer

> __What is going on in this code block?__
>
> The training dataset is a random sample of $\theta \times 100$% of the records. We use `rand(...)` combined with a `Set` to generate unique training indices. The test dataset contains the remaining $(1-\theta) \times 100$% of records, computed via `setdiff(...)`.

The code block below returns `(training_dataset, test_dataset)`.

In [7]:
training_dataset, test_dataset = let
    
    # initialize -
    training_dataset = Vector{Tuple{Vector{Float32}, OneHotVector{UInt32}}}();
    test_dataset = Vector{Tuple{Vector{Float32}, OneHotVector{UInt32}}}();
    number_of_samples = length(converted_dataset);
    all_index_set = range(1, stop=number_of_samples, step=1) |> Set{Int64};

    # generate a set of random indices for training and testing -
    random_training_index_set = Set{Int64}();
    while length(random_training_index_set) ≤ number_of_training_samples
        random_index = rand(1:number_of_samples);
        push!(random_training_index_set, random_index);
    end
    random_test_index_set = setdiff(all_index_set, random_training_index_set); # the rest of the indices will be used for testing
    
    # populate the training set -
    random_training_index_vector = random_training_index_set |> collect;
    for i ∈ eachindex(random_training_index_vector)
        index = random_training_index_vector[i];
        push!(training_dataset, converted_dataset[index]);
    end
    
    # populate the test set -
    random_test_index_vector = random_test_index_set |> collect;
    for i ∈ eachindex(random_test_index_vector)
        index = random_test_index_vector[i];
        push!(test_dataset, converted_dataset[index]);
    end

    # return -
    training_dataset, test_dataset;
end;

In [8]:
test_dataset

39983-element Vector{Tuple{Vector{Float32}, OneHotVector{UInt32}}}:
 ([-0.9541789, -0.906901, -1.1317072, 0.7853122, 0.17931679, 0.0, 0.99535596, 0.46564302, 0.16758014, -0.71067965, -1.1815649, 1.7869309, 1.6768152, 1.3529587, 0.743809, 0.59931844, -0.12229791, 1.0, 1.0, -0.6397446], [1, 0, 0])
 ([-0.9541789, -0.5649604, 0.78111833, -0.7198685, -0.3522989, 1.0, -0.77338755, -0.7483246, -0.4044337, 1.7103976, 0.78396094, 1.0152273, -0.74133325, -0.97730464, -0.53211546, -1.0359002, -0.011222625, 5.0, -1.0, -0.31653506], [0, 0, 1])
 ([-0.9541789, -0.25800794, -0.19747679, 2.5178058, -1.9371154, -1.0, -0.013171344, 0.19587244, 0.39638567, -0.41179842, -0.2086209, 1.0152273, -0.2577036, 1.3529587, 0.743809, 1.8257324, 1.8313097, 1.0, 1.0, 0.54893583], [1, 0, 0])
 ([-1.2329279, 3.6537037, 2.309019, -0.606212, 1.7240114, 1.0, -1.668914, -0.14134078, -1.3196558, -0.5330101, 2.2930741, -0.14232822, -1.224963, -0.7183865, -0.21313433, 0.19051377, -1.0211143, 4.0, 1.0, 1.0355159], [1, 0, 0])
 (

___

## Task 1: Construct and Train a Feedforward Credit Score Classifier
In this task, we'll construct and train a feedforward neural network model using the training records encoded in `training_dataset::Vector{Tuple{Vector{Float32}, OneHotVector{UInt32}}}`.

> __How is the model structured?__
>
> We use [the `Flux.jl` machine learning library](https://github.com/FluxML/Flux.jl) to build the model. The network has three layers: an input layer of size `number_of_features` $\times$ `number_of_hidden_nodes` with `tanh` activation, a hidden layer of size `number_of_hidden_nodes` $\times$ `number_of_classes` with `tanh` activation, and a `softmax` output layer. The model is built using [the `Chain` function](https://fluxml.ai/Flux.jl/stable/reference/models/layers/#Flux.Chain) with [`Dense` layers](https://fluxml.ai/Flux.jl/stable/reference/models/layers/#Flux.Dense).

The code block below returns `model::Chain`.

In [9]:
Flux.@layer MyFluxNeuralNetworkModel  trainable=(input, middle, hidden);
MyModel() = MyFluxNeuralNetworkModel(
    Chain(
        input = Dense(number_of_features, number_of_hidden_nodes, tanh_fast),  # layer 1
        hidden = Dense(number_of_hidden_nodes, number_of_classes, tanh_fast),  # layer 2
        output = NNlib.softmax)                                                 # layer 3 (output layer)
);
model = MyModel().chain;

How many parameters does our model have? For a network with $L$ layers where layer $i$ has $m_i$ nodes and layer $0$ has $m_0 = d_{in}$ input features, the total number of parameters is ([L8c lecture](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026)):

> __How many parameters does the model have?__
>
> Each layer $i$ has a weight matrix $\mathbf{W}_i \in \mathbb{R}^{m_i \times m_{i-1}}$ and a bias vector $\mathbf{b}_i \in \mathbb{R}^{m_i}$, giving $m_i(m_{i-1} + 1)$ parameters per layer. Summing over all $L$ layers:
>
> $$\text{Number of parameters} = \sum_{i=1}^{L} m_{i}(m_{i-1}+1) = \underbrace{\sum_{i=1}^{L} m_{i}\,m_{i-1}}_{\text{weights}} + \underbrace{\sum_{i=1}^{L} m_{i}}_{\text{biases}}$$
>
> The `softmax` output layer has no learnable parameters and does not contribute to the count.

The code block below counts the parameters in each `Dense` layer and returns `parameter_count::Int64`.

In [10]:
parameter_count = let
    dense_layers = [model.layers.input, model.layers.hidden]; # softmax has no learnable parameters
    total = 0;
    for (i, layer) ∈ enumerate(dense_layers)
        m_prev = size(layer.weight, 2); # m_{i-1}: input dimension
        m_i    = size(layer.weight, 1); # m_i: output dimension
        n_weights = m_i * m_prev;       # weight matrix entries
        n_biases  = m_i;                # bias vector entries
        n_params  = m_i * (m_prev + 1); # m_i*(m_{i-1}+1) per the L8c formula
        println("Layer $i ($(m_prev) → $(m_i)): $(n_weights) weights + $(n_biases) biases = $(n_params) parameters");
        total += n_params;
    end
    println("─────────────────────────────────────────");
    println("Total: $(total) parameters");
    total
end

Layer 1 (20 → 4096): 81920 weights + 4096 biases = 86016 parameters
Layer 2 (4096 → 3): 12288 weights + 3 biases = 12291 parameters
─────────────────────────────────────────
Total: 98307 parameters


98307

> __What loss function do we use?__
>
> We use [logit cross-entropy](https://fluxml.ai/Flux.jl/stable/reference/models/losses/#Flux.Losses.logitcrossentropy) as the loss function for this multiclass classification problem:
>
> $$\mathcal{L}(\theta) = -\frac{1}{N}\sum_{i=1}^{N}\sum_{j=1}^{C} y_{ij}\log(p_{ij}(\theta))$$
>
> where the outer sum is over all $N$ training examples, the inner sum is over the $C$ classes, $y_{ij}$ is the one-hot encoded label for example $i$, and $p_{ij}(\theta)$ is the predicted probability of example $i$ belonging to class $j$. Note: `logitcrossentropy` applies `softmax` internally, and since the model already applies `softmax` as its output layer, the effective loss applies `softmax` twice. This acts as an implicit regularizer and is the default behavior in this lab.

The code block below defines the `loss(ŷ, y)::Function`.

In [11]:
# TODO: Uncomment below to setup the loss function -
loss(ŷ, y) = Flux.Losses.logitcrossentropy(ŷ, y; agg = mean); # loss for training multiclass classifiers, what is the agg?

The unweighted loss treats all classes equally, but the training data is imbalanced: `Standard` accounts for roughly 56% of records, `Poor` for 28%, and `Good` for only 16%. A model minimizing unweighted cross-entropy can achieve high accuracy by rarely or never predicting `Good`, since errors on that class contribute little to the total loss.

> __How does inverse class frequency weighting address class imbalance?__
>
> We assign each class $c$ a weight $w_c \propto \frac{1}{N_c}$, where $N_c$ is the number of training examples in class $c$. The weights are normalized so their mean equals 1. The weighted cross-entropy loss is:
>
> $$\mathcal{L}_{\text{weighted}}(\theta) = -\frac{1}{N}\sum_{i=1}^{N}\sum_{j=1}^{C} w_j \, y_{ij}\log\hat{p}_{ij}(\theta)$$
>
> Classes with fewer examples receive larger weights, so the optimizer treats errors on rare classes as more costly. This discourages the model from collapsing all predictions to the majority class.

The code block below computes `class_weights` from the training data and defines `weighted_loss`.

In [12]:
class_weights, weighted_loss = let
    counts = zeros(Int, number_of_classes);
    for (_, y) ∈ training_dataset
        counts[argmax(y)] += 1;
    end
    w = Float32(length(training_dataset)) ./ (Float32(number_of_classes) .* Float32.(counts));
    w = w ./ sum(w) .* Float32(number_of_classes); # normalize: mean weight = 1
    # model already applies softmax, so ŷ is a probability vector — no extra softmax needed
    wloss(ŷ, y) = -sum(w .* Float32.(y) .* log.(ŷ .+ eps(Float32)));
    w, wloss
end

(Float32[0.9433181, 0.51637036, 1.5403115], var"#wloss#wloss##0"(Core.Box(Float32[0.9433181, 0.51637036, 1.5403115])))

> __What optimizer do we use?__
>
> We use [gradient descent with momentum](https://en.wikipedia.org/wiki/Stochastic_gradient_descent#Momentum). At each iteration $k$, the velocity $v_k$ and parameter vector $\theta_k \in \mathbb{R}^{p}$ are updated as:
>
> $$v_{k+1} = \beta\, v_{k} - \lambda\, \nabla_{\theta}\mathcal{L}(\theta_{k})$$
>
> $$\theta_{k+1} = \theta_{k} + v_{k+1}$$
>
> where $\lambda > 0$ is the learning rate, $\beta \in [0, 1)$ is the momentum parameter, and $\nabla_{\theta}\mathcal{L}(\theta_{k})$ is the gradient of the loss with respect to the parameters at iteration $k$. The momentum term $\beta\, v_{k}$ accumulates past gradients, allowing the optimizer to build speed in consistent directions and dampen oscillations. The optimizer state is stored in `opt_state` and passed to the training loop.

The code block below returns `opt_state`, the optimizer state used during training.

In [13]:
λ = 0.05; # TODO: maybe change the learning rate (default: 0.61)?
β = 0.10; # TODO: maybe change the momentum parameter (default: 0.10)?
opt_state = Flux.setup(Momentum(λ,β), model);

> __What is going on in this code block?__
>
> We train the model using `Flux.train!` with gradient descent with momentum. Set `use_class_weights = false` to minimize the unweighted logit cross-entropy loss; set `use_class_weights = true` to use the inverse-frequency weighted cross-entropy loss defined above. We run `number_of_epochs` passes through the training data, saving a checkpoint after each epoch. If `should_we_train = false`, a pre-trained model is loaded from disk instead.

The code block below returns `trainedmodel::Chain`.

In [14]:
trainedmodel = let

    should_we_train = false; # TODO: set this flag to {true | false}
    use_class_weights = true; # TODO: set to true to use inverse class frequency weighting
    current_loss = use_class_weights ? weighted_loss : loss;

    # training loop -
    localmodel = model; # make a local copy of the model
    if (should_we_train == true)
        for i = 1:number_of_epochs    
            
            # train the model - check out the do block notion: https://docs.julialang.org/en/v1/base/base/#do
            Flux.train!(localmodel, training_dataset, opt_state) do m, x, y
                current_loss(m(x), y)
            end
        
            # let the user know how we are doing -
            if (rem(i,2) == 0)
                @show "Epoch $i of $number_of_epochs completed" # print the epoch number
            end
        
            # save the state of the model, in case something happens. We can reload from this state
            jldsave(joinpath(_PATH_TO_DATA, "tmp-model-training-checkpoint.jld2"), model_state = Flux.state(localmodel))    
        end
    else
        # if we don't train: load up a previous model
        model_state = JLD2.load(joinpath(_PATH_TO_DATA, "Model-training-N$(number_of_hidden_nodes).jld2"), "model_state");
        Flux.loadmodel!(localmodel, model_state);
    end
    localmodel; # return the *trained* model (either freshly trained or loaded from a saved file)
end

Chain(
  input = Dense(20 => 4096, tanh_fast),  # 86_016 parameters
  hidden = Dense(4096 => 3, tanh_fast),  # 12_291 parameters
  output = NNlib.softmax,
)                   # Total: 4 arrays, 98_307 parameters, 384.215 KiB.

__Discussion:__ In this task, we built and trained a two-layer feedforward network with `tanh_fast` activations in the hidden layers and a `softmax` output layer.

* The output layer uses `softmax` while the hidden layers use `tanh_fast`. Why is `softmax` the appropriate activation for the output of a multiclass classifier? What mathematical property does `softmax` enforce that `tanh_fast` does not?
* The network has `2^10 = 1024` hidden nodes but only `20` input features, giving 24,579 total parameters trained on approximately 60k examples. This is a heavily overparameterized setting. Look ahead at the training and test accuracy in Task 2 — does the concern about overfitting appear to be warranted here, and why or why not?

___

## Task 2: Does the model generalize?
In this task, we'll check the feedforward network's ability to generalize, i.e., calculate how well the network classifies records it has not seen.

> __What is going on in this code block?__
>
> We pass each feature vector $\mathbf{x}$ into `trainedmodel`, take the `argmax` of the output probability vector to get the predicted class, convert it to a one-hot vector, and compare it to the true label $y$. The fraction of matching predictions over all samples gives the classification accuracy.

The code block below prints the fraction of correctly classified records in the training dataset.

In [15]:
let 
    S_training = 0;
    number_digit_array = [1,2,3];
    for i ∈ eachindex(training_dataset)
        x = training_dataset[i][1];
        y = training_dataset[i][2];
        ŷ = trainedmodel(x) |> z-> argmax(z) |> z-> number_digit_array[z] |> z-> onehot(z, number_digit_array)
        y == ŷ ? S_training += 1 : nothing
    end
    correct_prediction_training = (S_training/number_of_training_samples)*100 |> x-> round(x, digits=2);
    println("Correct classification on the training data: $(correct_prediction_training)%");
end

Correct classification on the training data: 64.08%


> __What is going on in this code block?__
>
> We repeat the same accuracy calculation on the `test_dataset` to measure how well the model generalizes to records it has not seen during training.

The code block below prints the fraction of correctly classified records in the test dataset.

In [16]:
let
    S_testing = 0;
    number_digit_array = [1,2,3];
    for i ∈ eachindex(test_dataset)
        x = test_dataset[i][1];
        y = test_dataset[i][2];
        ŷ = trainedmodel(x) |> z-> argmax(z) |> z-> number_digit_array[z] |> z-> onehot(z, number_digit_array)
        y == ŷ ? S_testing += 1 : nothing
    end
    correct_prediction_test = (S_testing/number_of_test_samples)*100 |> x-> round(x, digits=2);
    println("Correct classification on the test data: $(correct_prediction_test)%");
end

Correct classification on the test data: 64.39%


The overall accuracy gives one number but hides where the model makes mistakes. The confusion matrix reveals which specific classes are being confused with which others.

> __What is the confusion matrix for a multiclass problem?__
>
> For $C$ classes, the confusion matrix is a $C \times C$ matrix $\mathbf{M}$ where entry $M_{ij}$ counts the number of records that actually belong to class $i$ but were predicted as class $j$. Diagonal entries $M_{ii}$ are correct classifications; off-diagonal entries reveal specific misclassification patterns. For our three-class problem ($C = 3$), the rows and columns correspond to `Poor`, `Standard`, and `Good`.

The code block below returns `confusion_matrix::DataFrame`.

In [17]:
confusion_matrix = let

    # initialize -
    CM = zeros(Int, number_of_classes, number_of_classes);
    class_labels = ["Poor", "Standard", "Good"];

    # fill the confusion matrix using the test dataset -
    for i ∈ eachindex(test_dataset)
        x = test_dataset[i][1];
        y = test_dataset[i][2];
        actual    = argmax(y);               # true class index (row)
        predicted = argmax(trainedmodel(x)); # predicted class index (col)
        CM[actual, predicted] += 1;
    end

    # return as a labeled DataFrame (rows: actual class, columns: predicted class) -
    df = DataFrame(CM, ["Predicted: Poor", "Predicted: Standard", "Predicted: Good"]);
    insertcols!(df, 1, :Actual => class_labels);
    df
end

Row,Actual,Predicted: Poor,Predicted: Standard,Predicted: Good
,String,Int64,Int64,Int64
1,Poor,8924,729,1896
2,Standard,6127,10807,4357
3,Good,243,884,6016


Raw counts depend on class size, making it hard to compare performance across classes. Row-normalizing the confusion matrix expresses each entry as a percentage of all actual records in that class.

> __What does row normalization show?__
>
> Each entry $M_{ij}$ is divided by the row total $\sum_{k} M_{ik}$, giving the fraction of actual class $i$ records predicted as class $j$. The diagonal entries become per-class recall (e.g., what fraction of actual `Good` customers were correctly identified), and off-diagonal entries show where each class leaks into other predicted classes.

The code block below returns `confusion_matrix_pct::DataFrame`.

In [18]:
confusion_matrix_pct = let
    CM = Matrix(confusion_matrix[:, 2:end]);      # extract numeric block from raw count DataFrame
    class_labels = ["Poor", "Standard", "Good"];
    row_sums = sum(CM, dims=2);                   # total actual records per class
    CM_pct = round.(100.0 .* CM ./ row_sums, digits=1);
    df = DataFrame(CM_pct, ["Predicted: Poor (%)", "Predicted: Standard (%)", "Predicted: Good (%)"]);
    insertcols!(df, 1, :Actual => class_labels);
    df
end

Row,Actual,Predicted: Poor (%),Predicted: Standard (%),Predicted: Good (%)
,String,Float64,Float64,Float64
1,Poor,77.3,6.3,16.4
2,Standard,28.8,50.8,20.5
3,Good,3.4,12.4,84.2


__Discussion:__ The training accuracy and test accuracy are both approximately 62%.

* The gap between training accuracy and test accuracy is small. What does this tell us about whether the model is overfitting? If the training accuracy were 98% but test accuracy were 65%, what would that indicate about the model?
* The percentage confusion matrix shows that per-class recall varies substantially: `Poor` is correctly identified ~73% of the time, `Standard` ~69%, but `Good` only ~22%. In a credit scoring application, classifying a `Poor` risk customer as `Good` (false positive) carries a different cost than classifying a `Good` risk customer as `Poor` (false negative). Given the low recall for `Good`, which direction of misclassification is the model most prone to, and how might the lender respond to this asymmetry in practice?
* The current model uses `number_of_hidden_nodes = 2^10 = 1024` hidden nodes, `number_of_epochs = 25` training epochs, and a learning rate of `λ = 0.40`. Experiment with these values — try reducing the hidden layer width (e.g., `2^8 = 256` or `2^9 = 512`), increasing the number of epochs, or adjusting `λ`. Does changing the model structure or training configuration improve overall accuracy or the recall for `Good`?
___

## Summary
This lab constructed, trained, and evaluated a feedforward neural network to classify consumer credit scores into three categories: `Poor`, `Standard`, and `Good`.

> __Key Takeaways__
>
> * **Data preprocessing determines model inputs:** Categorical variables must be encoded as integers and continuous features must be z-score standardized before training. The processed records are packaged as `(feature vector, one-hot label)` tuples, the format `Flux.jl` expects directly.
> * **Feedforward networks learn from labeled examples:** We trained a two-layer network by minimizing logit cross-entropy loss over multiple epochs using gradient descent with momentum. The model maps 20 input features to a probability distribution over three credit score classes.
> * **Generalization is measured on held-out data:** We evaluated the trained model on a separate test set to assess whether it generalizes beyond the training data. Similar training and test accuracy indicates the model is not overfitting.

Similar classification accuracy on training and test data confirms that this feedforward architecture generalizes well to new consumer credit records.
___